In [ ]:
# ============================================================================
# CELL 1: Install Packages
# ============================================================================
!pip install torch transformers==4.47.1 accelerate==0.26.1 scikit-learn numpy pandas tqdm

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import re
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
# ============================================================================
# CELL 3: Load Data
# ============================================================================
train_path = "train_data (1).json"
test_path = "test_data_subtask_1 (1).json"

with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(test_path, 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nValidity distribution:\n{train_df['validity'].value_counts()}")
print(f"\nPlausibility distribution:\n{train_df['plausibility'].value_counts()}")

# Analyze 4 conditions
conditions = train_df.groupby(['plausibility', 'validity']).size()
print(f"\n4-Condition Distribution (Original):")
print(conditions)

Training samples: 960
Test samples: 191

Validity distribution:
validity
False    480
True     480
Name: count, dtype: int64

Plausibility distribution:
plausibility
False    486
True     474
Name: count, dtype: int64

4-Condition Distribution (Original):
plausibility  validity
False         False       246
              True        240
True          False       234
              True        240
dtype: int64


In [ ]:
# ============================================================================
# CELL 4: PROPER Train/Val Split BEFORE Augmentation
# ============================================================================
print("\n" + "="*80)
print("Step 1: Split ORIGINAL Data First (Prevent Data Leakage)")
print("="*80)

# Create condition column for stratification
train_df['condition'] = (
    train_df['plausibility'].astype(str) + '_' +
    train_df['validity'].astype(str)
)

# Split ORIGINAL data first
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=train_df['condition'].values
)

train_df_original = train_df.iloc[train_indices].copy()
val_df_original = train_df.iloc[val_indices].copy()

print(f"\nOriginal data split:")
print(f"  Training: {len(train_df_original)}")
print(f"  Validation: {len(val_df_original)}")
print(f"\n CRITICAL: Augmentation will ONLY be applied to training data")
print(f"   Validation stays ORIGINAL to test true generalization")


Step 1: Split ORIGINAL Data First (Prevent Data Leakage)

Original data split:
  Training: 816
  Validation: 144

 CRITICAL: Augmentation will ONLY be applied to training data
   Validation stays ORIGINAL to test true generalization


In [ ]:
# ============================================================================
# CELL 5: Template-Based Augmentation (TRAINING ONLY)
# ============================================================================
print("\n" + "="*80)
print("Step 2: Augment ONLY Training Data")
print("="*80)

class TemplateAugmenter:
    """Generate variations that preserve logical structure"""

    def __init__(self):
        self.entity_pools = {
            'vehicles': ['car', 'bicycle', 'motorcycle', 'truck', 'bus', 'train', 'airplane', 'boat'],
            'buildings': ['house', 'building', 'tower', 'castle', 'barn', 'shed', 'cabin', 'mansion'],
            'animals': ['dog', 'cat', 'horse', 'bird', 'fish', 'rabbit', 'lion', 'tiger'],
            'people': ['person', 'human', 'individual', 'citizen', 'student', 'teacher', 'worker', 'doctor'],
            'objects': ['book', 'table', 'chair', 'tool', 'device', 'machine', 'instrument', 'gadget'],
            'food': ['fruit', 'vegetable', 'meat', 'drink', 'meal', 'snack', 'dish', 'dessert'],
            'nature': ['tree', 'plant', 'flower', 'rock', 'mountain', 'river', 'ocean', 'forest'],
        }

    def identify_entities(self, text):
        """Extract main entities"""
        entities = []
        text_lower = text.lower()

        for category, entity_list in self.entity_pools.items():
            for entity in entity_list:
                if entity in text_lower:
                    entities.append((entity, category))

        return list(set(entities))[:2]  # Max 2 entities

    def replace_entities(self, text, old_entities, new_entities):
        """Case-preserving replacement"""
        result = text

        for (old, _), (new, _) in zip(old_entities, new_entities):
            result = re.sub(re.escape(old), new, result, flags=re.IGNORECASE)
            result = re.sub(re.escape(old.capitalize()), new.capitalize(), result)

        return result

    def generate_variations(self, text, validity, plausibility, n_variations=2):
        """Generate n variations with different entities"""
        variations = []
        entities = self.identify_entities(text)

        if not entities:
            return variations

        for i in range(n_variations):
            new_entities = []

            for old_entity, old_category in entities:
                # Pick new entity from same category
                available = [e for e in self.entity_pools[old_category] if e != old_entity]
                if available:
                    new_entity = random.choice(available)
                    new_entities.append((new_entity, old_category))
                else:
                    new_entities.append((old_entity, old_category))

            if len(new_entities) == len(entities):
                new_text = self.replace_entities(text, entities, new_entities)

                if new_text != text:
                    variations.append({
                        'syllogism': new_text,
                        'validity': validity,
                        'plausibility': plausibility
                    })

        return variations

# Augment ONLY training data
augmenter = TemplateAugmenter()
augmented_train = []
variation_count = 0

for idx, row in tqdm(train_df_original.iterrows(), total=len(train_df_original), desc="Augmenting training"):
    # Add original
    augmented_train.append(row.to_dict())

    # Generate 2 variations
    variations = augmenter.generate_variations(
        row['syllogism'],
        row['validity'],
        row['plausibility'],
        n_variations=2
    )

    augmented_train.extend(variations)
    variation_count += len(variations)

train_df_augmented = pd.DataFrame(augmented_train)
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n Training data augmentation:")
print(f"  Original training: {len(train_df_original)}")
print(f"  Generated variations: {variation_count}")
print(f"  Total training: {len(train_df_augmented)}")
print(f"  Expansion: {len(train_df_augmented)/len(train_df_original):.2f}x")

print(f"\n Validation data (UNCHANGED):")
print(f"  Validation samples: {len(val_df_original)}")
print(f"  Status: NO AUGMENTATION (true generalization test)")


Step 2: Augment ONLY Training Data


Augmenting training: 100%|██████████| 816/816 [00:00<00:00, 7361.74it/s]


✓ Training data augmentation:
  Original training: 816
  Generated variations: 1224
  Total training: 2040
  Expansion: 2.50x

✓ Validation data (UNCHANGED):
  Validation samples: 144
  Status: NO AUGMENTATION (true generalization test)


In [ ]:
# ============================================================================
# CELL 6: Oversample Implausible in Training Only
# ============================================================================
print("\n" + "="*80)
print("Step 3: Oversample Implausible (Training Only)")
print("="*80)

# Separate by plausibility in augmented training data
plausible_train = train_df_augmented[train_df_augmented['plausibility'] == True]
implausible_train = train_df_augmented[train_df_augmented['plausibility'] == False]

print(f"\nBefore oversampling:")
print(f"  Plausible: {len(plausible_train)}")
print(f"  Implausible: {len(implausible_train)}")

# Oversample implausible 2x
implausible_oversampled = pd.concat([implausible_train, implausible_train], ignore_index=True)

# Combine
train_df_final = pd.concat([plausible_train, implausible_oversampled], ignore_index=True)
train_df_final = train_df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter oversampling:")
print(f"  Total training: {len(train_df_final)}")

# Add dummy counterfactuals
train_df_final['counterfactual'] = train_df_final['syllogism']
val_df_original['counterfactual'] = val_df_original['syllogism']

# Final data
train_df = train_df_final
val_df = val_df_original

conditions_train = train_df.groupby(['plausibility', 'validity']).size()
conditions_val = val_df.groupby(['plausibility', 'validity']).size()

print(f"\nFinal 4-Condition Distribution:")
print(f"  TRAINING (augmented + oversampled):")
for cond, count in conditions_train.items():
    print(f"    {cond}: {count}")
print(f"  VALIDATION (original, no augmentation):")
for cond, count in conditions_val.items():
    print(f"    {cond}: {count}")

print(f"\n KEY: Validation = original data → Tests true generalization")


Step 3: Oversample Implausible (Training Only)

Before oversampling:
  Plausible: 1035
  Implausible: 1005

After oversampling:
  Total training: 3045

Final 4-Condition Distribution:
  TRAINING (augmented + oversampled):
    (False, False): 1054
    (False, True): 956
    (True, False): 529
    (True, True): 506
  VALIDATION (original, no augmentation):
    (False, False): 37
    (False, True): 36
    (True, False): 35
    (True, True): 36

 KEY: Validation = original data → Tests true generalization


In [ ]:
# ============================================================================
# CELL 7: Dataset Class
# ============================================================================
class SyllogismDataset(Dataset):
    def __init__(self, texts, counterfactuals, validity_labels, plausibility_labels, tokenizer, max_length=512):
        self.texts = texts
        self.counterfactuals = counterfactuals
        self.validity_labels = validity_labels
        self.plausibility_labels = plausibility_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded_orig = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        encoded_cf = self.tokenizer(
            self.counterfactuals[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_orig': encoded_orig['input_ids'].squeeze(0),
            'attention_mask_orig': encoded_orig['attention_mask'].squeeze(0),
            'input_ids_cf': encoded_cf['input_ids'].squeeze(0),
            'attention_mask_cf': encoded_cf['attention_mask'].squeeze(0),
            'validity_label': torch.tensor(self.validity_labels[idx], dtype=torch.long),
            'plausibility_label': torch.tensor(self.plausibility_labels[idx], dtype=torch.long)
        }

In [ ]:
# ============================================================================
# CELL 8: Focal Loss
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
     inputs = inputs.float()  # ensure float32
     ce_loss = F.cross_entropy(inputs, targets, reduction='none')
     ce_loss = torch.clamp(ce_loss, max=100.0)  # prevent exp overflow
     p_t = torch.exp(-ce_loss)
     focal_loss = (1 - p_t) ** self.gamma * ce_loss

     if self.alpha is not None:
        alpha_t = self.alpha[targets]
        focal_loss = alpha_t * focal_loss

     if self.reduction == 'mean':
        return focal_loss.mean()
     elif self.reduction == 'sum':
        return focal_loss.sum()
     else:
        return focal_loss

In [ ]:
# ============================================================================
# CELL 9: Gradient Reversal Layer
# ============================================================================
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super(GradientReversalLayer, self).__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

In [ ]:
# ============================================================================
# CELL 10: MLA-CI Model
# ============================================================================
class MLACIModel(nn.Module):
    def __init__(self, model_name, lambda_adv=1.0, dropout=0.2):
        super(MLACIModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(
              model_name,
              output_hidden_states=True,
             low_cpu_mem_usage=False  # Force eager loading, disables accelerate's lazy materialization
             )
        self.encoder = self.encoder.float()  # Cast AFTER loading, not during

        hidden_size = self.encoder.config.hidden_size
        self.combined_size = hidden_size * 3
        self.grl = GradientReversalLayer(lambda_=lambda_adv)

        self.validity_classifier = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

        self.plausibility_adversary = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def extract_multi_layer_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states

        layer_2 = hidden_states[2][:, 0, :]
        layer_6 = hidden_states[6][:, 0, :]
        layer_neg2 = hidden_states[-2][:, 0, :]

        combined = torch.cat([layer_2, layer_6, layer_neg2], dim=1)
        return combined.float()

    def forward(self, input_ids, attention_mask, return_features=False):
        features = self.extract_multi_layer_features(input_ids, attention_mask)

        validity_logits = self.validity_classifier(features)

        reversed_features = self.grl(features)
        plausibility_logits = self.plausibility_adversary(reversed_features)

        if return_features:
            return validity_logits, plausibility_logits, features
        return validity_logits, plausibility_logits

In [ ]:
# ============================================================================
# CELL 11: Data Preparation (Using Pre-Split Data)
# ============================================================================
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use the already-split data (NO SECOND SPLIT!)
# train_df = augmented + oversampled training data
# val_df = original validation data (no augmentation)

print(f"\n Using Pre-Split Data:")
print(f"  Training: {len(train_df)} samples (augmented + oversampled)")
print(f"  Validation: {len(val_df)} samples (original, no augmentation)")

# Prepare training data
train_texts = train_df['syllogism'].tolist()
train_cf = train_df['counterfactual'].tolist()
train_validity = train_df['validity'].astype(int).values
train_plausibility = train_df['plausibility'].astype(int).values

# Prepare validation data
val_texts = val_df['syllogism'].tolist()
val_cf = val_df['counterfactual'].tolist()
val_validity = val_df['validity'].astype(int).values
val_plausibility = val_df['plausibility'].astype(int).values

# Create datasets
train_dataset = SyllogismDataset(train_texts, train_cf, train_validity, train_plausibility, tokenizer)
val_dataset = SyllogismDataset(val_texts, val_cf, val_validity, val_plausibility, tokenizer)

# Create dataloaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Batch size: {BATCH_SIZE}")

# Initialize model
model = MLACIModel(MODEL_NAME, lambda_adv=1.0, dropout=0.2)
model = model.to(device)

print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================================
# CELL 12: Training Configuration
# ============================================================================
EPOCHS = 4  # Reduced since we have 5-6x more data
LEARNING_RATE = 2e-5
LAMBDA_ADV = 1.0
GRADIENT_ACCUMULATION_STEPS = 4

# Focal Loss parameters
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = None

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = (len(train_loader) // GRADIENT_ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

focal_criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)
ce_criterion = nn.CrossEntropyLoss()

print(f"\n TRAINING CONFIGURATION:")
print(f"  Epochs: {EPOCHS} (reduced due to 5-6x more data)")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Lambda adversarial: {LAMBDA_ADV}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Total steps: {total_steps}")
print(f"\n FOCAL LOSS:")
print(f"  Gamma: {FOCAL_GAMMA}")
print(f"\n STRATEGY:")
print(f"  1. Template-based augmentation (2x variations)")
print(f"  2. Stratified oversampling (2x implausible)")
print(f"  3. Focal loss (hard example focus)")
print(f"  4. Adversarial debiasing")
print(f"  → Total data expansion: ~5-6x original")


 TRAINING CONFIGURATION:
  Epochs: 4 (reduced due to 5-6x more data)
  Learning rate: 2e-05
  Lambda adversarial: 1.0
  Gradient accumulation: 4
  Effective batch size: 16
  Total steps: 760

 FOCAL LOSS:
  Gamma: 2.0

 STRATEGY:
  1. Template-based augmentation (2x variations)
  2. Stratified oversampling (2x implausible)
  3. Focal loss (hard example focus)
  4. Adversarial debiasing
  → Total data expansion: ~5-6x original


In [ ]:
# ============================================================================
# CELL 13: Training Loop
# ============================================================================
def train_epoch_focal(model, loader, optimizer, scheduler, lambda_adv, accumulation_steps=4):
    model.train()
    total_loss = 0
    total_validity_loss = 0
    total_plausibility_loss = 0
    correct = 0
    total = 0

    optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(loader, desc="Training")):
        input_ids_orig = batch['input_ids_orig'].to(device)
        attention_mask_orig = batch['attention_mask_orig'].to(device)
        validity_labels = batch['validity_label'].to(device)
        plausibility_labels = batch['plausibility_label'].to(device)

        validity_logits_orig, plausibility_logits_orig = model(input_ids_orig, attention_mask_orig)

        loss_validity = focal_criterion(validity_logits_orig, validity_labels)
        loss_plausibility = ce_criterion(plausibility_logits_orig, plausibility_labels)

        loss = (loss_validity + lambda_adv * loss_plausibility) / accumulation_steps

        loss.backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        total_loss += (loss.item() * accumulation_steps)
        total_validity_loss += loss_validity.item()
        total_plausibility_loss += loss_plausibility.item()

        predictions = torch.argmax(validity_logits_orig, dim=1)
        correct += (predictions == validity_labels).sum().item()
        total += validity_labels.size(0)

    accuracy = correct / total
    return (total_loss / len(loader), total_validity_loss / len(loader),
            total_plausibility_loss / len(loader), accuracy)

def evaluate_detailed(model, loader, plausibility_labels_full):
    model.eval()

    all_predictions = []
    all_validity = []
    all_plausibility = []

    idx = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids_orig = batch['input_ids_orig'].to(device)
            attention_mask_orig = batch['attention_mask_orig'].to(device)
            validity_labels = batch['validity_label'].to(device)
            batch_size = validity_labels.size(0)

            validity_logits_orig, _ = model(input_ids_orig, attention_mask_orig)
            predictions = torch.argmax(validity_logits_orig, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_validity.extend(validity_labels.cpu().numpy())

            for i in range(batch_size):
                all_plausibility.append(plausibility_labels_full[idx])
                idx += 1

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    all_predictions = np.array(all_predictions)
    all_validity = np.array(all_validity)
    all_plausibility = np.array(all_plausibility)

    overall_acc = accuracy_score(all_validity, all_predictions)

    # Per-condition accuracy
    conditions = {}
    for plaus in [0, 1]:
        for valid in [0, 1]:
            mask = (all_plausibility == plaus) & (all_validity == valid)
            if mask.sum() > 0:
                acc = accuracy_score(all_validity[mask], all_predictions[mask])
                plaus_name = "Plausible" if plaus == 1 else "Implausible"
                valid_name = "Valid" if valid == 1 else "Invalid"
                conditions[f"{plaus_name}_{valid_name}"] = acc

    # Calculate content effects
    intra_plaus = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Plausible_Invalid', 0))
    intra_implaus = abs(conditions.get('Implausible_Valid', 0) - conditions.get('Implausible_Invalid', 0))
    intra_effect = (intra_plaus + intra_implaus) / 2

    cross_valid = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Implausible_Valid', 0))
    cross_invalid = abs(conditions.get('Plausible_Invalid', 0) - conditions.get('Implausible_Invalid', 0))
    cross_effect = (cross_valid + cross_invalid) / 2

    total_content_effect = (intra_effect + cross_effect) / 2

    return overall_acc, total_content_effect, conditions

# Clear cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Training loop
print("\n" + "="*80)
print("Training MLA-CI v4: Template Augmentation + Stratified + Focal")
print("="*80)

best_combined_score = 0
best_model_state = None

for epoch in range(EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")

    train_loss, val_loss, plaus_loss, train_acc = train_epoch_focal(
        model, train_loader, optimizer, scheduler, LAMBDA_ADV, GRADIENT_ACCUMULATION_STEPS
    )

    val_acc, val_content_effect, val_conditions = evaluate_detailed(
        model, val_loader, val_plausibility
    )

    combined_score = val_acc * 100 / (1 + np.log(1 + val_content_effect * 100))

    print(f"\n Training:")
    print(f"  Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}")
    print(f"  Validity Loss (Focal): {val_loss:.4f}")
    print(f"  Plausibility Loss (Adv): {plaus_loss:.4f}")

    print(f"\n Validation:")
    print(f"  Accuracy: {val_acc*100:.2f}%")
    print(f"  Content Effect: {val_content_effect*100:.2f}%")
    print(f"  Combined Score: {combined_score:.2f}")

    print(f"\n Per-condition:")
    for cond, acc in sorted(val_conditions.items()):
        print(f"  {cond:25s}: {acc*100:.2f}%")

    if combined_score > best_combined_score:
        best_combined_score = combined_score
        best_model_state = model.state_dict().copy()
        torch.save(model.state_dict(), 'best_model_v4.pt')
        print(f"\n NEW BEST! Score: {combined_score:.2f}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Load best model
model.load_state_dict(best_model_state)

# Final evaluation
final_acc, final_content_effect, final_conditions = evaluate_detailed(
    model, val_loader, val_plausibility
)

print(f"\n{'='*80}")
print(f"BEST VALIDATION RESULTS")
print(f"{'='*80}")
print(f"  Accuracy: {final_acc*100:.2f}%")
print(f"  Content Effect: {final_content_effect*100:.2f}%")
print(f"  Combined Score: {best_combined_score:.2f}")
print(f"\n Per-condition:")
for cond, acc in sorted(final_conditions.items()):
    print(f"  {cond:25s}: {acc*100:.2f}%")


Training MLA-CI v4: Template Augmentation + Stratified + Focal

Epoch 1/4


Evaluating: 100%|██████████| 36/36 [00:07<00:00,  4.81it/s]



 Training:
  Loss: 0.8407 | Accuracy: 0.5586
  Validity Loss (Focal): 0.1721
  Plausibility Loss (Adv): 0.6687

 Validation:
  Accuracy: 68.75%
  Content Effect: 20.11%
  Combined Score: 16.98

 Per-condition:
  Implausible_Invalid      : 67.57%
  Implausible_Valid        : 77.78%
  Plausible_Invalid        : 80.00%
  Plausible_Valid          : 50.00%

 NEW BEST! Score: 16.98

Epoch 2/4


Evaluating: 100%|██████████| 36/36 [00:07<00:00,  4.77it/s]



 Training:
  Loss: 0.7717 | Accuracy: 0.8309
  Validity Loss (Focal): 0.1022
  Plausibility Loss (Adv): 0.6695

 Validation:
  Accuracy: 77.08%
  Content Effect: 13.36%
  Combined Score: 21.03

 Per-condition:
  Implausible_Invalid      : 62.16%
  Implausible_Valid        : 88.89%
  Plausible_Invalid        : 71.43%
  Plausible_Valid          : 86.11%

 NEW BEST! Score: 21.03

Epoch 3/4


...


 Training:
  Loss: 0.7252 | Accuracy: 0.9412
  Validity Loss (Focal): 0.0437
  Plausibility Loss (Adv): 0.6815

 Validation:
  Accuracy: 81.94%
  Content Effect: 7.79%
  Combined Score: 25.82

 Per-condition:
  Implausible_Invalid      : 78.38%
  Implausible_Valid        : 88.89%
  Plausible_Invalid        : 82.86%
  Plausible_Valid          : 77.78%

 NEW BEST! Score: 25.82

Epoch 4/4


Evaluating: 100%|██████████| 36/36 [00:07<00:00,  4.79it/s]



 Training:
  Loss: 0.6878 | Accuracy: 0.9783
  Validity Loss (Focal): 0.0172
  Plausibility Loss (Adv): 0.6706

 Validation:
  Accuracy: 79.17%
  Content Effect: 10.57%
  Combined Score: 22.95

 Per-condition:
  Implausible_Invalid      : 75.68%
  Implausible_Valid        : 83.33%
  Plausible_Invalid        : 85.71%
  Plausible_Valid          : 72.22%


Evaluating: 100%|██████████| 36/36 [00:07<00:00,  4.80it/s]


BEST VALIDATION RESULTS
  Accuracy: 79.17%
  Content Effect: 10.57%
  Combined Score: 25.82

 Per-condition:
  Implausible_Invalid      : 75.68%
  Implausible_Valid        : 83.33%
  Plausible_Invalid        : 85.71%
  Plausible_Valid          : 72.22%


In [ ]:
################################################################################
# CELL 13.5: Download Checkpoint (SIMPLE VERSION)
################################################################################

from google.colab import files

print("\n Downloading checkpoint...")
files.download('best_model_v4.pt')

print(" Checkpoint downloaded! Save this file!")

In [ ]:
# ============================================================================
# CELL 14: Test Predictions
# ============================================================================
print("\n" + "="*80)
print("Generating Test Predictions")
print("="*80)

test_texts = test_df['syllogism'].tolist()
test_cf = test_texts
test_validity_dummy = np.zeros(len(test_texts), dtype=int)
test_plausibility_dummy = np.zeros(len(test_texts), dtype=int)

test_dataset = SyllogismDataset(test_texts, test_cf, test_validity_dummy, test_plausibility_dummy, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
all_predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids_orig'].to(device)
        attention_mask = batch['attention_mask_orig'].to(device)

        validity_logits, _ = model(input_ids, attention_mask)
        predictions = torch.argmax(validity_logits, dim=1)
        all_predictions.extend(predictions.cpu().numpy().tolist())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nGenerated {len(all_predictions)} predictions")
print(f"  Valid: {sum(all_predictions)} ({100*sum(all_predictions)/len(all_predictions):.1f}%)")
print(f"  Invalid: {len(all_predictions) - sum(all_predictions)} ({100*(len(all_predictions)-sum(all_predictions))/len(all_predictions):.1f}%)")


Generating Test Predictions


Predicting: 100%|██████████| 48/48 [00:09<00:00,  4.80it/s]


Generated 191 predictions
  Valid: 100 (52.4%)
  Invalid: 91 (47.6%)


In [ ]:
# ============================================================================
# CELL 15: Create Submission
# ============================================================================
submission = []
for test_id, pred in zip(test_df['id'], all_predictions):
    submission.append({
        "id": test_id,
        "validity": bool(pred)
    })

with open('predictions.json', 'w') as f:
    json.dump(submission, f, indent=2)

import zipfile
with zipfile.ZipFile('predictions.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('predictions.json')

print("\n Submission created:")
print("  - predictions.json")
print("  - predictions.zip")


 Submission created:
  - predictions.json
  - predictions.zip


In [ ]:
# ============================================================================
# CELL 16: Final Summary
# ============================================================================
print("\n" + "="*80)
print("FINAL SUMMARY - MLA-CI v4")
print("="*80)

print(f"\n METHOD:")
print(f"  1. Split ORIGINAL data into train/val FIRST")
print(f"  2. Template augmentation on TRAINING only (2x)")
print(f"  3. Stratified oversampling on TRAINING only (2x implausible)")
print(f"  4. Focal Loss (γ={FOCAL_GAMMA}) + Adversarial (λ={LAMBDA_ADV})")
print(f"  5. Validation = ORIGINAL data (no augmentation)")

print(f"\n FIX APPLIED:")
print(f"   Previous v4: Augment THEN split → Data leakage")
print(f"     Val acc: 97.91% | Test acc: 76.96% | Gap: -21%")
print(f"   Fixed v4: Split THEN augment training only")
print(f"     Val acc: {final_acc*100:.2f}% | Expected realistic test performance")

print(f"\n VALIDATION PERFORMANCE:")
print(f"  Accuracy: {final_acc*100:.2f}%")
print(f"  Content Effect: {final_content_effect*100:.2f}%")
print(f"  Combined Score: {best_combined_score:.2f}")

print(f"\n PER-CONDITION:")
for cond, acc in sorted(final_conditions.items()):
    print(f"  {cond:25s}: {acc*100:.2f}%")

condition_accs = list(final_conditions.values())
std_dev = np.std(condition_accs) * 100
print(f"\n  Std Dev: {std_dev:.2f}%")

print(f"\n EXPECTED TEST PERFORMANCE:")
val_test_gap = 5  # Typical generalization gap
print(f"  Accuracy: ~{final_acc*100-val_test_gap:.1f}% (val - ~5%)")
print(f"  Content Effect: ~{final_content_effect*100:.1f}%")
print(f"  Combined Score: ~{best_combined_score-1.5:.1f}")

print(f"\n COMPARISON:")
print(f"  v3 (Stratified+Focal): Test Score = 26.00")
print(f"  v4 Fixed (+ Template): Expected = {best_combined_score-1.5:.1f}-{best_combined_score:.1f}")

print(f"\n DATA INTEGRITY CHECK:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Val is original data: ✓")
print(f"  No data leakage: ✓")

print(f"\n Ready for CodaBench!")
print(f"   File: predictions_v4_fixed.zip")
print(f"   Expected: Realistic test score ~{best_combined_score-1.5:.1f}-{best_combined_score:.1f}")
print("="*80)